# Week 2 Day 1 Lab - Evals and Observability

## Measuring an agent on UnderCover's harder claims

A new batch of 25 claims has arrived at UnderCover, and they're harder than the queue from Week 1. Many have long lists of items, with payments from airlines, hotels and other insurers to take off, and deductibles, limits and caps to apply.

We trace the agent from Week 1 in Langfuse and score its decisions against the handlers'. Then we run two experiments: the agent working out each payout itself, and the agent with a payout tool.

## 4 parts:

PART 1: The harder claims

PART 2: Tracing the agent with Langfuse

PART 3: Scoring an agent

PART 4: Experiments in Langfuse

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls GPT-6 Luna through OpenRouter and costs a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week2day1free.ipynb">week2day1free.ipynb</a>.<br/><br/>
            You'll also need a free Langfuse account, which we set up below. Each experiment takes 2 or 3 minutes to run. For a quicker run, set <code>TINY = True</code> in Part 4 to use 5 claims in place of 25.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made in Week 1 in the `undercover` folder.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

## Connect to Langfuse

Langfuse is an open-source platform for tracing and evaluating LLM applications. You can host it yourself, or use Langfuse Cloud, which has a free Hobby plan.

1. Sign up at [cloud.langfuse.com](https://cloud.langfuse.com) and create a project
2. In the project's settings, go to API Keys and create a new key
3. Add the three values it shows to your `.env` file: `LANGFUSE_SECRET_KEY`, `LANGFUSE_PUBLIC_KEY` and `LANGFUSE_BASE_URL`

In [ ]:
load_dotenv(override=True)
langfuse = get_client()

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## Part 1: The harder claims

The harder claims are in `data/evals/harder`, with their own policy records and the handlers' decisions. The policy wording is the same as before.

In [ ]:
HARDER = Path("../data/evals/harder")


def load(name):
    """Read one of the JSON files of the harder claims."""
    return json.loads((HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "items": len(claim["items"]),
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

Similar mix of decisions

In [ ]:
data["decision"].value_counts()

### A burglary in Rome

Here's one of the longer claims.

In [ ]:
rome = [claim for claim in claims if claim["claim_id"] == "CLM-1203"][0]

print(rome["summary"])
print(rome["story"])
pd.DataFrame(rome["items"])

This is how the handler worked out the payout.

In [ ]:
decision = labels["CLM-1203"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

## Part 2: Tracing the agent with Langfuse

This is the agent from Week 1 Day 2 with two changes. The customer's policy record goes in the prompt with the claim, so its only tool is `record_decision`. We also turn Luna's reasoning off with `reasoning_effort="none"`, which makes each call faster and cheaper.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

The tool's docstring is in Google style. With `parse_docstring=True`, LangChain turns each line under `Args` into the description of that argument.

The rationale comes before the decision, so the model writes its reasons before it decides. As in UnderCover's claims system, a decision can't be changed once it's recorded, and an approval needs a payout above zero.

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="none", timeout=120, max_retries=5)

baseline_agent = create_agent(model=luna, tools=[record_decision], system_prompt=f"{INSTRUCTIONS}\n{policy_wording}\n")

Langfuse's `CallbackHandler` sends a trace of everything LangChain does to Langfuse. We pass it to the agent in the `config`.

In [ ]:
langfuse_handler = CallbackHandler()

recorded.clear()
result = baseline_agent.invoke(
    {"messages": [{"role": "user", "content": make_user_prompt(rome)}]},
    config={"callbacks": [langfuse_handler]},
)

for message in result["messages"][1:]:
    message.pretty_print()

Now open **Tracing** in Langfuse. The trace shows each call to the model with its prompt, its reply, the tokens and the time it took, and the tool call in between.

## Part 3: Scoring an agent

In Week 1 a claim was right when the decision matched the handler's and the payout was within a dollar. We still count that, but not every mistake costs UnderCover the same.

### The confusion matrix

A confusion matrix compares an agent's decisions with the right answers. Each row is the handler's decision and each column is the agent's. The claims on the diagonal are the ones where they agree, and each cell off the diagonal is a kind of mistake:

- **A wrongful decline:** the agent declines a claim the handler approved or referred. The customer is let down and may complain.
- **A wrongful payment:** the agent approves a claim the handler declined or referred. UnderCover pays money it doesn't owe.
- **A needless referral:** the agent refers a claim the handler decided. A handler has to spend time on it.

The confusion matrix compares decisions only, so an approval with the wrong payout still sits on the diagonal. The cost of mistakes below counts the wrong amounts.

### Precision and recall

Precision and recall come from the same matrix, with an approval as the positive:

- **Precision:** of the claims the agent approved, the share the handler approved.
- **Recall:** of the claims the handler approved, the share the agent approved.

### The cost of mistakes

A cost matrix gives each kind of mistake a cost in dollars. A wrongful payment costs the amount paid, and a wrong amount on a correct approval costs the difference.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">The UnderCover business has more tolerance for false positives (we cover someone when we shouldn't) than false negatives (we decline someone when we should have covered them), because false negatives damage our reputation and customer loyalty. Our evals take this into account.<br/><br/>
            Agree the figures in the cost matrix with the business before you run your evals.
            </span>
        </td>
    </tr>
</table>

### Kappa

Accuracy can flatter an agent. 14 of the 25 claims are approvals, so an agent that approves everything gets 56% of the decisions right. Cohen's kappa corrects for this. It measures how much more often the agent agrees with the handlers than it would by chance, based on how often each of them picks each decision. A kappa of 1 is perfect agreement and 0 is no better than chance.

In [ ]:
def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


approve_everything = pd.DataFrame({"handler": data["decision"], "agent": "approve"})

print(f"Approve everything gets a kappa of {kappa(approve_everything):.2f}")

Here's the confusion matrix for approving everything. Every claim lands in the approve column.

In [ ]:
def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()


show_confusion_matrix(approve_everything, "Approve everything")

### The straight-through rate

The straight-through rate is the share of claims the agent decides without referring them to a handler. It includes approvals over $1,000, which still need a handler's sign-off. An agent that refers every claim never pays or declines a claim wrongly. But its straight-through rate is 0%, and each of its 19 needless referrals costs $50. The handlers refer 6 of the 25 claims, so their rate is 76%.

## Part 4: Experiments in Langfuse

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Datasets and experiments</h2>
            <span style="color:#118AB2;">A Langfuse dataset holds test inputs with their expected outputs. An experiment runs a task on each item of a dataset and scores the result with evaluators. An item evaluator scores one item and a run evaluator scores the whole run.<br/><br/>
            Langfuse keeps every run of a dataset with its scores and traces, so you can compare runs side by side.
            </span>
        </td>
    </tr>
</table>

### The dataset

We upload the harder claims as a dataset, with the handler's decision as the expected output. Each item's id is the claim id, so running this cell again updates the items in place.

In [ ]:
DATASET = "harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"]},
        metadata={"tiny": label["tiny"]},
    )

By default we run all 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

`N_TIMES` is the number of times each experiment runs. The agent's answers vary from run to run, and more runs give steadier scores. With more than one run, each confusion matrix adds up the claims from every run, and the comparison at the end averages the scores.

In [ ]:
TINY = False
N_TIMES = 3

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims")

### The task and the evaluators

The task runs an agent on one claim. If the agent finishes without recording a decision, the claim would go to a handler, so we count it as a referral.

In [ ]:
def decide(agent, claim):
    """Run an agent on one claim and return its decision, with the cost of its model calls."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost}

`score_claim` is the item evaluator and `score_run` is the run evaluator. `score_table` puts each claim's answer beside the handler's.

In the run's scores, `decision_accuracy` is the share of claims on the diagonal of the confusion matrix. `accuracy` also needs the payout within a dollar.

In [ ]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, and the cost of its mistake."""
    return [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]

### Experiment 1: the baseline

The agent works out each payout itself, following section 10 of the policy wording. Langfuse runs the claims one at a time, so each run takes about 2 minutes.

In [ ]:
def baseline_task(*, item, **kwargs):
    """Run the baseline agent on one claim from the dataset."""
    return decide(baseline_agent, item.input)


baseline_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    baseline = langfuse.run_experiment(
        name="Baseline",
        description="The agent works out each payout itself",
        data=items,
        task=baseline_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(baseline.format())
    baseline_tables.append(score_table(baseline.item_results).assign(run=run))

In [ ]:
baseline_table = pd.concat(baseline_tables, ignore_index=True)
show_confusion_matrix(baseline_table, "Baseline")

Here are the claims it got wrong, with the agent's rationale.

In [ ]:
baseline_table[~baseline_table["right"]]

### Experiment 2: a payout tool

The payout rules in section 10 are fixed business logic, and business logic belongs in code. `week2day1_payout.py` has UnderCover's payout rules as a Python function, `work_out_payout`, and we give it to the agent as a tool. The agent still judges whether the claim is covered, which lines count and what counts as one item.

In [ ]:
@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

A tool is still a Python function, so we can try it ourselves. Here's a suitcase and three T-shirts on an Economy policy, where the airline has paid $100.

In [ ]:
calculate_payout.invoke({
    "policy_id": "POL-2201",
    "benefit": "baggage",
    "lines": [
        {"description": "Hard-shell suitcase", "amount": 239.99, "kind": "item"},
        {"description": "Three T-shirts", "amount": 74.97, "quantity": 3, "kind": "item"},
    ],
    "paid_by_others": 100,
})

The agent gets the tool and one more line in its instructions.

In [ ]:
PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")


def payout_tool_task(*, item, **kwargs):
    """Run the agent with the payout tool on one claim from the dataset."""
    return decide(payout_agent, item.input)

In [ ]:
payout_tool_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    with_payout_tool = langfuse.run_experiment(
        name="Payout tool",
        description="The agent works out each payout with the payout tool",
        data=items,
        task=payout_tool_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(with_payout_tool.format())
    payout_tool_tables.append(score_table(with_payout_tool.item_results).assign(run=run))

In [ ]:
payout_tool_table = pd.concat(payout_tool_tables, ignore_index=True)
show_confusion_matrix(payout_tool_table, "Payout tool")

In [ ]:
payout_tool_table[~payout_tool_table["right"]]

### Experiment 3: the short wording with the payout tool

The same agent as Experiment 2, with the concise policy wording from Week 1 Day 4.

In [ ]:
policy_wording_short = Path("../data/policy_wording_short.md").read_text(encoding="utf-8")

short_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording_short}\n")


def short_task(*, item, **kwargs):
    """Run the agent with the short wording and the payout tool on one claim."""
    return decide(short_agent, item.input)

In [ ]:
short_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    short = langfuse.run_experiment(
        name="Short wording + payout tool",
        description="The concise policy wording, with the payout tool",
        data=items,
        task=short_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(short.format())
    short_tables.append(score_table(short.item_results).assign(run=run))

In [ ]:
short_table = pd.concat(short_tables, ignore_index=True)
show_confusion_matrix(short_table, "Short wording + payout tool")

In [ ]:
short_table[~short_table["right"]]

### Full wording against short wording

In [ ]:
def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()


tables = {"Full wording": payout_tool_table, "Short wording": short_table}
pd.DataFrame([average_scores(t) for t in tables.values()], index=tables.keys()).round(3)

### Scoring again with a different WRONGFUL_DECLINE

The tables hold every answer, so we recompute the cost of mistakes without calling the model again.

In [ ]:
def rescore(table):
    """Recompute mistake_cost with the current WRONGFUL_DECLINE."""
    costs = [mistake_cost({"decision": r.agent, "payout": r.agent_payout}, {"decision": r.handler, "payout": r.handler_payout}) for r in table.itertuples()]
    return table.assign(mistake_cost=costs)


for WRONGFUL_DECLINE in [500, 1500, 5000, 10000]:
    print(f"\nWRONGFUL_DECLINE = {WRONGFUL_DECLINE}")
    print(pd.DataFrame([average_scores(rescore(t)) for t in tables.values()], index=tables.keys())[["cost_of_mistakes", "decision_accuracy", "llm_cost"]].round(3))

WRONGFUL_DECLINE = 1500

In [ ]:
print("The baseline got either decision or amount wrong for:", len(baseline_table[~baseline_table["right"]]) )
print("The payout tool version got either decision or amount wrong for:", len(payout_tool_table[~payout_tool_table["right"]]) )


Open the traces for these claims in Langfuse to see the lines the agent passed to the tool and the calculation that came back.

### Comparing the experiments

In [ ]:
def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()


pd.DataFrame([average_scores(baseline_table), average_scores(payout_tool_table)], index=["Baseline", "Payout tool"]).round(3)

In Langfuse, open **Datasets** and then `harder-claims`. Select the runs to compare them side by side, claim by claim.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: on Week 1 Day 4 we wrote a concise version of the policy wording, in <code>data/policy_wording_short.md</code>. Run a third experiment with the concise wording and the payout tool, and compare it with the full wording in Langfuse. Does it keep the accuracy? Is it cheaper or faster?<br/><br/>
            Then change <code>WRONGFUL_DECLINE</code> and score the runs again. Does it change which wording you would choose?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## Cleaning up Langfuse

This last step is optional. Set `CLEAN_UP = True` and run the cell to delete every trace in the Langfuse project your keys belong to, along with the experiments built on them, and the claims in the `harder-claims` dataset. This can't be undone. Only do it in a project you made for this course.

The empty `harder-claims` dataset stays, and running Part 4 again fills it. The run scores (accuracy, kappa and the rest) stay too. Langfuse's free plan allows 50 deletions of trace data a day, and each score would take one.

In [ ]:
def all_trace_ids():
    """Every trace id in the Langfuse project, collected from its observations a page at a time."""
    trace_ids = set()
    cursor = None
    while True:
        page = langfuse.api.observations.get_many(fields="core", limit=1000, cursor=cursor)
        trace_ids.update(observation.trace_id for observation in page.data)
        cursor = page.meta.cursor
        if not cursor:
            return list(trace_ids)


CLEAN_UP = False

if CLEAN_UP:
    trace_ids = all_trace_ids()
    for start in range(0, len(trace_ids), 1000):
        langfuse.api.trace.delete_multiple(trace_ids=trace_ids[start:start + 1000])

    dataset_items = langfuse.get_dataset(DATASET).items
    for item in dataset_items:
        langfuse.api.dataset_items.delete(id=item.id)

    print(f"Deleted {len(trace_ids)} traces and {len(dataset_items)} dataset items")